# 02 — Honest baseline: 5-fold patient cross-validation (Google Colab)

Every patient is tested exactly once, at each magnification (40x, 100x, 200x, 400x), with two groupings:
- **`patient_id`** — the usual way (e.g. `14-22549AB` and `14-22549CD` count as different patients)
- **`patient_base`** — the patient *number* only, so IDs that are probably the same person stay together

Total: 2 groupings × 4 magnifications × 5 folds = **40 training runs, about 1–1.5 hours on a T4.**
Every fold is saved to Drive as soon as it finishes. If Colab disconnects, just **Runtime → Run all** again and it continues where it stopped.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Keep this tab open and your laptop awake.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/breakhis-results'

## 1. Get the latest code

In [ ]:
import os, sys
REPO = '/content/breakhis-reliability'
if not os.path.exists(REPO):
    !git clone -q https://github.com/amubakekengbam/breakhis-reliability.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -e .
if f'{REPO}/src' not in sys.path:
    sys.path.insert(0, f'{REPO}/src')
import breakhis; print('breakhis', breakhis.__version__, 'ready')

## 2. Data (skips the download if it's already unpacked in this session)

In [ ]:
import subprocess
URL = 'http://www.inf.ufpr.br/vri/databases/BreaKHis_v1.tar.gz'
ARCHIVE = '/content/BreaKHis_v1.tar.gz'
DATA_DIR = '/content/BreaKHis_v1'

def remote_size(url):
    out = subprocess.run(['curl', '-sIL', url], capture_output=True, text=True).stdout
    sizes = [int(l.split(':')[1]) for l in out.lower().splitlines() if l.startswith('content-length')]
    return sizes[-1] if sizes else None

if os.path.exists(f'{DATA_DIR}/histology_slides'):
    print('Data already unpacked - nothing to do.')
else:
    expected = remote_size(URL)
    print('Expected size:', f'{expected/1e9:.2f} GB' if expected else 'unknown')
    !wget -c -q --show-progress -O {ARCHIVE} {URL}
    have = os.path.getsize(ARCHIVE)
    print(f'Downloaded: {have/1e9:.2f} GB')
    if expected and have != expected:
        raise RuntimeError('Download incomplete - run this cell again; it resumes where it stopped.')
    !tar -xzf {ARCHIVE} -C /content/
    print('Done:', os.path.exists(f'{DATA_DIR}/histology_slides'))

## 3. Patient-ID check (should show 81 IDs, 70 numbers, and `14-13412` under two tumor types)

In [ ]:
from breakhis.data import build_metadata, id_report
df = build_metadata(DATA_DIR)
print(len(df), 'images')
id_report(df)

## 4. Run Experiment 2 (~1–1.5 hours)

In [ ]:
!PYTHONPATH={REPO}/src python -m breakhis.experiments.cv --data_dir {DATA_DIR} --out_dir {RESULTS}/cv --epochs 5

## 5. Results

In [ ]:
import pandas as pd
s = pd.read_csv(f'{RESULTS}/cv/cv_summary.csv')
cols = ['grouping','magnification','accuracy','balanced_accuracy','balanced_accuracy_ci_low','balanced_accuracy_ci_high',
        'auroc','patient_balanced_accuracy','n_test_patients','shared_numbers_max']
display(s[cols].round(3))
print('Per tumor type, 200x, grouped by patient number:')
display(pd.read_csv(f'{RESULTS}/cv/patient_base/200x/per_subtype.csv'))

## 6. Send to Claude
Screenshot or copy the two tables above. Or download the whole `breakhis-results/cv` folder from Drive as a zip and attach it.